<a href="https://colab.research.google.com/github/ColmTalbot/HSFIndia2026/blob/main/population-inference/gwtc4_population.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spectral Siren Population Inference on GWTC-4

The fourth gravitational-wave transient catalog [GWTC-4](https://arxiv.org/pdf/2508.18082) includes all compact binary coalescences observed during Advanced LIGO/Virgo's first three oberving runs.

"Spectral siren" cosmology (see, e.g., [Ezquiaga & Holz](https://arxiv.org/abs/2202.08240)) uses features in the black hole mass spectrum to constrain the distance-redshift relation.

`GWPopulation` builds upon [`Bilby`](git.ligo.org/lscsoft/bilby) ([Talbot _et al._ (2025)](https://joss.theoj.org/papers/10.21105/joss.07753)) to provide simple, modular, user-friendly, population inference.
`wcosmo` provides optimized cosmology functionality for `GWPopulation`.

There are many [implemented models](https://colmtalbot.github.io/gwpopulation/_autosummary/gwpopulation.models.html#module-gwpopulation.models) and an example of defining custom models is included below.
In this example we use:

- A mass distribution in primary mass and mass ratio from Talbot & Thrane (2018) ([arXiv:1801:02699](https://arxiv.org/abs/1801.02699)). This is equivalent to the `PowerLaw + Peak` model used in LVK analyses without the low-mass smoothing for computational efficiency.
- Half-Gaussian + isotropic spin tilt distribution from Talbot & Thrane (2017) ([arXiv:1704.08370](https://arxiv.org/abs/1704.08370)).
- A truncated Gaussian distribution as used in[arXiv:2508.18082](https://arxiv.org/pdf/2508.18082).
- Redshift evolution model as in Fishbach+ (2018) ([arXiv:1805.10270](https://arxiv.org/abs/1805.10270)).
- A variable Flat Lambda CDM cosmology using a modified version of the approximation in [arXiv:1111.6396](https://arxiv.org/abs/1111.6396) as implemented in `wcosmo`.

For more information on `GWPopulation` see the [git repository](https://github.com/ColmTalbot/gwpopulation), [documentation](https://colmtalbot.github.io/gwpopulation/).


## Setup

If you're using colab.research.google.com you will want to choose a GPU-accelerated runtime (I'm going to use a T4 GPU).

"runtime"->"change runtime type"->"Hardware accelerator = GPU"

## Install some needed packages

All of the packages we need for this notebook other than `GWPopulation` (and it's dependencies) are already installed in Google colab, so we just need the following installation command.

In [ ]:
!pip install gwpopulation

## Imports

Import the packages required for the script.
We also set the backend for array operations to `jax` which allows us to take advantage of just-in-time (jit) compilation in addition to GPU-parallelisation when available.

In [ ]:
import bilby as bb
import gwpopulation as gwpop
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from bilby.core.prior import PriorDict, Uniform
from gwpopulation.experimental.cosmo_models import CosmoModel
from gwpopulation.experimental.jax import JittedLikelihood
from wcosmo import disable_units, Planck15

disable_units()
gwpop.set_backend("jax")

xp = gwpop.utils.xp

%matplotlib inline

## Load data

We're using the posteriors and sensitivity injections from the GWTC-4 data release in a pre-processed format.

These files were produced by [`gwpopulation-pipe`](https://docs.ligo.org/ratesAndPopulations/gwpopulation_pipe) to reduce the many GB of posterior sample files to a pair of ~100Mb files.

This file includes all of the confidently identified binary black hole mergers and no potentially neutron star containing binaries.

We also need to modify some of the stored parameters as we are going to include a cosmological fit so we will fit detector frame masses and luminosity distance.

The data files can be found [here](https://drive.google.com/drive/folders/1wyfR6sYvYVdBefF9_vrVTp0Btu03OlzL?usp=sharing).
The original data can be found at [zenodo:5546663](https://zenodo.org/records/5546663), [zenodo:6513631](https://zenodo.org/records/6513631), and [zenodo:16053484](https://zenodo.org/records/16053484) along with citation information and the sensitivity injections are available at[zenodo:16740128](https://zenodo.org/records/16740128).

In [ ]:
!gdown https://drive.google.com/uc?id=1qkhkXD1Y_tm-YW74CuklcfiBkorJ0OLi
!gdown https://drive.google.com/uc?id=1nmXFr_yVcxVWO4D6c3cs8y1M7JgSFzDo

In [ ]:
posteriors = pd.read_pickle("gwtc4_posteriors.pkl")
data = pd.read_pickle("gwtc4_injections_dict.pkl")

for post in posteriors:
    post["luminosity_distance"] = Planck15.luminosity_distance(post["redshift"])
    post["mass_1_detector"] = post.pop("mass_1") * (1 + post["redshift"])
    post["prior"] /= Planck15.dDLdz(post["redshift"]) * (1 + post["redshift"])
    del post["redshift"]

data["luminosity_distance"] = Planck15.luminosity_distance(data["redshift"])
data["mass_1_detector"] = data.pop("mass_1") * (1 + data["redshift"])
data["prior"] /= Planck15.dDLdz(data["redshift"]) * (1 + data["redshift"])
del data["redshift"]

Let's make some plots to visualize the data.

We will include in all the plots both the found injections and the per-event posteriors.
We should expect the found injections to fully cover the per-event posterior samples to ensure reliable population inference.

In [ ]:
plt.scatter(data["mass_1_detector"], data["mass_ratio"], s=1, alpha=0.1)

for post in posteriors:
    plt.scatter(post["mass_1_detector"], post["mass_ratio"], s=1, alpha=0.1)
plt.xscale("log")
plt.xlabel("$m_1$ [detector frame]")
plt.ylabel("$q = m_2 / m_1$")
plt.show()
plt.close()

fig, ax = plt.subplots(subplot_kw={"projection": "polar"})
plt.scatter(
    -np.arccos(data["cos_tilt_1"]),
    data["a_1"],
    s=1,
    alpha=0.1,
)
plt.scatter(
    np.arccos(data["cos_tilt_2"]),
    data["a_2"],
    s=1,
    alpha=0.1,
)

for post in posteriors:
    plt.scatter(
        -np.arccos(post["cos_tilt_1"]),
        post["a_1"],
        s=1,
        alpha=0.1,
    )
    plt.scatter(
        np.arccos(post["cos_tilt_2"]),
        post["a_2"],
        s=1,
        alpha=0.1,
    )
ax.set_theta_offset(np.pi / 2)

plt.show()
plt.close()

plt.hist(np.log10(data["luminosity_distance"]), bins=100, density=True, histtype="step")

for post in posteriors:
    plt.hist(np.log10(post["luminosity_distance"]), bins=100, density=True, histtype="step")
plt.xlabel("$\\log(d_L [Mpc])$")
plt.show()
plt.close()

## Define some models and the likelihood

We need to define `Bilby` `Model` objects for the numerator and denominator independently as these cache some computations interally.

Note that we are using a [`CosmoModel`](https://colmtalbot.github.io/gwpopulation/autoapi/gwpopulation/experimental/cosmo_models/index.html#gwpopulation.experimental.cosmo_models.CosmoModel), this in an experimental feature and so the specific API may change in future, but the basic funcionality should be stable.
We create a model that can infer the three parameters of [`wCDM`](https://wcosmo.readthedocs.io/en/latest/api/_autosummary/wcosmo.astropy.FlatwCDM.html).

The [`HyperparameterLikelihood`](https://colmtalbot.github.io/gwpopulation/autoapi/gwpopulation/hyperpe/index.html#gwpopulation.hyperpe.HyperparameterLikelihood) is equivalent to marginalising over the local merger rate, with a uniform-in-log prior.
The posterior for the merger rate can be recovered in post-processing.

We provide:
- `posteriors`: a list of `pandas` DataFrames.
- `hyper_prior`: our population model, as defined above.
- `selection_function`: anything which evaluates the selection function.

We can also provide:
- `conversion_function`: this converts between the parameters we sample in and those needed by the model, e.g., for sampling in the mean and variance of the beta distribution.
- `max_samples`: the maximum number of samples to use from each posterior, this defaults to the length of the shortest posterior.

### Define a custom spin model

The base model for spin magnitudes used in GWTC-4 isn't available in `GWPopulation` directly.
However, defining new models is straightforward!

The most important thing to match is the expected function signature.
`GWPopulation` performs introspection on the function to determine which variables to pass to each model.
- The first argument must be called `dataset` and should be a dictionary with keys given by the parameters to fit.
- The remaining arguments are the names of the population parameters.
- The function should return an array containing the model evaluated at each of the points in `dataset`.

Since a truncated normal is a standard building block for population models, we can use the version shipped with `GWPopulation`.

In [ ]:
from gwpopulation.utils import powerlaw, truncnorm


def truncated_normal_spin_magnitude(
    dataset: dict[str, xp.ndarray],
    mu_chi: float,
    sigma_chi: float,
    amax: float,
) -> xp.ndarray:
    kwargs = dict(mu=mu_chi, sigma=sigma_chi, low=0, high=amax)
    return (
        truncnorm(dataset["a_1"], **kwargs)
        * truncnorm(dataset["a_2"], **kwargs)
    )

In [ ]:
model = CosmoModel(
    model_functions=[
        gwpop.models.mass.two_component_primary_mass_ratio,
        truncated_normal_spin_magnitude,
        gwpop.models.spin.iid_spin_orientation_gaussian_isotropic,
        gwpop.models.redshift.PowerLawRedshift(cosmo_model="FlatwCDM"),
    ],
    cosmo_model="FlatwCDM",
)

vt = gwpop.vt.ResamplingVT(model=model, data=data, n_events=len(posteriors))

likelihood = gwpop.hyperpe.HyperparameterLikelihood(
    posteriors=posteriors,
    hyper_prior=model,
    selection_function=vt,
)

## Define our prior

The mass model has eight parameters that we vary that are described in arXiv:1801:02699. This model is sometimes referred to as "PowerLaw+Peak"

The spin magnitude model is a normal distribution with the usual parameterization, and the spin orientation model is a mixure of a uniform component and a truncated Gaussian that peaks at aligned spin. This combination is sometimes referred to as "Default".

For redshift we use a model that looks like

$$p(z) \propto \frac{d V_{c}}{dz} (1 + z)^{λ - 1}$$

Finally, we set priors on the three parameters of the `wCDM` model, `H0` (the Hubble constant), `Omega_{m0}` (the matter fraction of the Universe at current time), `w0` (the constant dark energy equation of state).

In [ ]:
priors = PriorDict()

# mass
priors["alpha"] = Uniform(minimum=-2, maximum=4, latex_label="$\\alpha$")
priors["beta"] = Uniform(minimum=-4, maximum=12, latex_label="$\\beta$")
priors["mmin"] = Uniform(minimum=2, maximum=10, latex_label="$m_{\\min}$")
priors["mmax"] = Uniform(minimum=150, maximum=400, latex_label="$m_{\\max}$")
priors["lam"] = Uniform(minimum=0, maximum=1, latex_label="$\\lambda_{m}$")
priors["mpp"] = Uniform(minimum=5, maximum=50, latex_label="$\\mu_{m}$")
priors["sigpp"] = Uniform(minimum=1, maximum=10, latex_label="$\\sigma_{m}$")
priors["gaussian_mass_maximum"] = Uniform(minimum=100, maximum=200, latex_label="$\\max_{g}$")
# spin
priors["amax"] = Uniform(minimum=0.5, maximum=1.0, latex_label="$a_{\\max}$")
priors["mu_chi"] = Uniform(
    minimum=-1, maximum=1, latex_label="$\\mu_{\\chi}$"
)
priors["sigma_chi"] = Uniform(minimum=0.05, maximum=2, latex_label="$\\sigma_{\\chi}$")
priors["mu_spin"] = Uniform(minimum=-1, maximum=1, latex_label="$\\mu$")
priors["xi_spin"] = Uniform(minimum=0, maximum=1, latex_label="$\\xi$")
priors["sigma_spin"] = Uniform(minimum=0.3, maximum=4, latex_label="$\\sigma$")

priors["H0"] = Uniform(minimum=20, maximum=200, latex_label="$H_0$")
priors["Om0"] = Uniform(minimum=0, maximum=1, latex_label="$\\Omega_{m,0}$")
priors["w0"] = Uniform(minimum=-1.5, maximum=-0.5, latex_label="$w_{0}$")
priors["lamb"] = Uniform(minimum=-1, maximum=10, latex_label="$\\lambda_{z}$")

## Just-in-time compile

We JIT compile the likelihood object before starting the sampler.
This is done using the `gwpopulation.experimental.jax.JittedLikelihood` class.

We then time the original likelihood object and the JIT-ed version.
Note that we do two evaluations for each object as the first evaluation must compile the likelihood and so takes longer. (In addition to the JIT compilation, `JAX` compiles GPU functionality at the first evaluation, but this is less extreme than the full JIT compilation.)

In [ ]:
parameters = priors.sample()
likelihood.log_likelihood_ratio(parameters)
%time print(likelihood.log_likelihood_ratio(parameters))
jit_likelihood = JittedLikelihood(likelihood)
%time print(jit_likelihood.log_likelihood_ratio(parameters))
%time print(jit_likelihood.log_likelihood_ratio(parameters))

## Run the sampler

We'll use the sampler `dynesty` and use a small number of live points to reduce the runtime (total runtime should be approximately 25 minutes on T4 GPUs via Google colab, on the A100 available with a Colab Pro subscription, this will be faster).
The settings here may not give publication quality results, a convergence test should be performed before making strong quantitative statements.

`bilby` times a single likelihood evaluation before beginning the run, however, this isn't well defined with JAX.

**Note:** sometimes this finds a high likelihood mode, likely due to [breakdowns in the approximation](https://arxiv.org/abs/2304.06138) used to estimate the likelihood. If you see `dlogz > -700`, you should interrupt the execution and restart.

In [ ]:
result = bb.run_sampler(
    likelihood=jit_likelihood,
    priors=priors,
    sampler="dynesty",
    nlive=100,
    label="cosmo",
    sample="acceptance-walk",
    verbose=True,
    naccept=5,
    resume=False,
    save="hdf5",
)

## Plot some posteriors

We can look at the posteriors on some of the parameters, here the cosmology parameters and the location of the mass peak and the redshift evolution.

We see that the value of the Hubble constant is strongly correlated with the location of the peak in the mass distribution as has been noted elsewhere.

We also include the values of the cosmology parameters reported in the `Planck15` cosmology for reference.

In [ ]:
_ = result.plot_corner(
    save=False,
    parameters=["H0", "Om0", "w0", "mpp", "lamb", "mmin", "mmax"],
    truths=[67.74, 0.3075, -1, np.nan, np.nan, np.nan, np.nan],
)

How does this look?

How can we improve this result?

The first thing to check is that we aren't being significantly biased by systematic error in our likelihood estimates.
`GWPopulation` provides a build in way to evaluate the total variance due to Monte Carlo in addition to the contribution from each of the indidivual events.

In [ ]:
import jax

func = jax.jit(likelihood.generate_extra_statistics)

full_posterior = pd.DataFrame([
    func(parameters) for parameters in result.posterior.to_dict(orient="records")
]).astype(float)
full_posterior.describe()

Now we have these values, we can visualize which variables are most strongly correlated with the variance and log likelihood and then make a corner plot including these variables.

In [ ]:
full_posterior[result.search_parameter_keys + ["log_likelihood", "variance"]].corr()

In [ ]:
pd.plotting.scatter_matrix(
    full_posterior[["H0", "mpp", "sigpp", "lamb", "log_likelihood", "variance"]],
    alpha=0.1,
)
plt.show()
plt.close()

These look reasonable, all the variances are around one, which shouldn't cause signficant biases.

The next most likely candidate for poor recorvery of cosmological parameters is using an insufficient model for the astrophysical population (see, e.g., [Farah _et al._ (2404.02210)](https://arxiv.org/abs/2404.02210)).

**Challenge**: implement a more sophisticated model for the masses and see if it improves the measurement of cosmological parameters.

_hint_: see the most recent [LVK population paper](https://arxiv.org/pdf/2508.18083) for inspiration.